<a href="https://colab.research.google.com/github/Not-kh-lily-23/pulsar-conformal-triage/blob/main/medlat_data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import tarfile
import pandas as pd
import xml.etree.ElementTree as ET
import numpy as np
from scipy.stats import skew, kurtosis
from google.colab import drive
drive.mount('/content/drive')
pulsar_tar='/content/drive/MyDrive/pulsar_project/data/raw/pulsars.tar.gz'
negative_tar='/content/drive/MyDrive/pulsar_project/data/raw/negatives_10K.tar.gz'
extract_base='/content/medlat_temp'
output_csv='/content/drive/MyDrive/pulsar_project/data/processed/medlat_10k.csv'
def hex_array(hex_str):
    clean_hex=''.join(hex_str.split())
    return np.frombuffer(bytes.fromhex(clean_hex),dtype=np.uint8).astype(float)
def process_file(file_path,label):
    try:
        tree=ET.parse(file_path)
        root=tree.getroot()
        profile_node=root.find('.//Section/Profile')
        snr_node=root.find('.//Section/SnrBlock/DataBlock')
        if profile_node is None or not profile_node.text or snr_node is None or not snr_node.text:
            return None
        profile_array=hex_array(profile_node.text)
        snr_array=hex_array(snr_node.text)
        return {
            "mean_profile":np.mean(profile_array),
            "std_profile":np.std(profile_array),
            "kurtosis_profile":kurtosis(profile_array,fisher=True),
            "skewness_profile":skew(profile_array),
            "mean_dmsnr":np.mean(snr_array),
            "std_dmsnr":np.std(snr_array),
            "kurtosis_dmsnr":kurtosis(snr_array,fisher=True),
            "skewness_dmsnr":skew(snr_array),
            "target":label
        }
    except Exception:
        return None
dataset=[]
for tar_path,label,name in [(pulsar_tar,1,"Pulsars"),(negative_tar,0,"Negatives")]:
    ext_path=os.path.join(extract_base,name)
    with tarfile.open(tar_path,'r:gz') as tar:
        tar.extractall(path=ext_path)
    print(f"Processing {name} XML files...")
    count=0
    for root_dir,_,files in os.walk(ext_path):
        for file in files:
            if file.endswith('.phcx') or file.endswith('.xml'):
                full_path=os.path.join(root_dir,file)
                row=process_file(full_path,label)
                if row:
                    dataset.append(row)
                count+=1
                if count%2000==0:
                    print(f"processed {count} files")
df=pd.DataFrame(dataset)
df.to_csv(output_csv,index=False)
print("dataset compiled")
print(f"total rows successfully processed - {len(df)}")
print(f"Pulsars (Class 1): {df['target'].sum()} | Noise (Class 0): {len(df) - df['target'].sum()}")
print(f"Saved to: {output_csv}")

Mounted at /content/drive


/tmp/ipykernel_1086/3895654043.py:43: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=ext_path)


Processing Pulsars XML files...
Processing Negatives XML files...
processed 2000 files
processed 4000 files
processed 6000 files
processed 8000 files
dataset compiled
total rows successfully processed - 11195
Pulsars (Class 1): 1196 | Noise (Class 0): 9999
Saved to: /content/drive/MyDrive/pulsar_project/data/processed/medlat_10k.csv


Previous code was a test run (used only 1 10k file) - below is important code

In [2]:
import os
import glob
import tarfile
import pandas as pd
import xml.etree.ElementTree as ET
import numpy as np
from scipy.stats import skew, kurtosis
from google.colab import drive
drive.mount('/content/drive',force_remount=True)
raw_dir='/content/drive/MyDrive/pulsar_project/data/raw/'
extract_base='/content/medlat_temp_full'
output_csv='/content/drive/MyDrive/pulsar_project/data/processed/medlat_full.csv'
def hex_array(hex_str):
    clean_hex=''.join(hex_str.split())
    return np.frombuffer(bytes.fromhex(clean_hex),dtype=np.uint8).astype(float)
def process_file(file_path,label):
    try:
        tree=ET.parse(file_path)
        root=tree.getroot()
        profile_node=root.find('.//Section/Profile')
        snr_node=root.find('.//Section/SnrBlock/DataBlock')
        if profile_node is None or not profile_node.text or snr_node is None or not snr_node.text:
            return None
        profile_array=hex_array(profile_node.text)
        snr_array=hex_array(snr_node.text)
        return {
            "mean_profile":np.mean(profile_array),
            "std_profile":np.std(profile_array),
            "kurtosis_profile":kurtosis(profile_array, fisher=True),
            "skewness_profile":skew(profile_array),
            "mean_dmsnr":np.mean(snr_array),
            "std_dmsnr":np.std(snr_array),
            "kurtosis_dmsnr":kurtosis(snr_array, fisher=True),
            "skewness_dmsnr":skew(snr_array),
            "target":label
        }
    except Exception:
        return None
dataset=[]
all_tars=glob.glob(os.path.join(raw_dir,'*.tar.gz'))
for tar_path in all_tars:
    filename=os.path.basename(tar_path)
    if 'pulsar' in filename.lower():
        label=1
        name="Pulsars"
    elif 'negative' in filename.lower():
        label=0
        name=filename.replace('.tar.gz','')
    else:
        continue
    print(f"extracting {name}")
    ext_path=os.path.join(extract_base, name)
    with tarfile.open(tar_path,'r:gz') as tar:
        tar.extractall(path=ext_path)
    print(f"processing {name} xml files")
    count=0
    for root_dir,_,files in os.walk(ext_path):
        for file in files:
            if file.endswith('.phcx') or file.endswith('.xml'):
                full_path=os.path.join(root_dir,file)
                row=process_file(full_path,label)
                if row:
                    dataset.append(row)
                count+=1
                if count%5000==0:
                    print(f"processed {count} files in {name}")
df=pd.DataFrame(dataset)
df.to_csv(output_csv,index=False)
print(f"number of rows that are successfully processed - {len(df)}")
print(f"Pulsars (Class 1): {df['target'].sum()} | Noise (Class 0): {len(df)-df['target'].sum()}")
print(f"Positive Class Ratio: {(df['target'].sum()/len(df))*100:.2f}%")
print(f"Saved to: {output_csv}")

Mounted at /content/drive
extracting Pulsars


/tmp/ipykernel_1086/3542308437.py:54: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=ext_path)


processing Pulsars xml files
extracting negatives_50K
processing negatives_50K xml files
processed 5000 files in negatives_50K
processed 10000 files in negatives_50K
extracting negatives_90K
processing negatives_90K xml files
processed 5000 files in negatives_90K
processed 10000 files in negatives_90K
extracting negatives_70K
processing negatives_70K xml files
processed 5000 files in negatives_70K
processed 10000 files in negatives_70K
extracting negatives_80K
processing negatives_80K xml files
processed 5000 files in negatives_80K
processed 10000 files in negatives_80K
extracting negatives_10K
processing negatives_10K xml files
processed 5000 files in negatives_10K
extracting negatives_60K
processing negatives_60K xml files
processed 5000 files in negatives_60K
processed 10000 files in negatives_60K
extracting negatives_40K
processing negatives_40K xml files
processed 5000 files in negatives_40K
extracting negatives_30K
processing negatives_30K xml files
processed 5000 files in negati